# RavenPack Smoke Test

Minimal check: connect to WRDS, pull a sample of RavenPack sentiment data for AAPL, verify the sentiment score formula.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import wrds
from dotenv import load_dotenv

load_dotenv(Path().resolve().parent / ".env")

db = wrds.Connection(
    wrds_username=os.environ["WRDS_USERNAME"],
    wrds_password=os.environ["WRDS_PASSWORD"],
)

def pg_sql(db, sql):
    """Query WRDS via psycopg2 directly (bypasses SQLAlchemy 2.x bug in wrds 3.x)."""
    cur = db.connection.connection.cursor()
    cur.execute(sql)
    df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    cur.close()
    return df

print("Connected to WRDS")

In [ ]:
# Look up AAPL's RavenPack entity ID
mapping = pg_sql(db, """
    SELECT DISTINCT rp_entity_id, entity_name, ticker
    FROM ravenpack_common.wrds_rpa_company_mappings
    WHERE ticker = 'AAPL'
""")
print(mapping)

In [ ]:
rp_entity_id = mapping["rp_entity_id"].iloc[0]

# Pull 50 rows from Q1 2007 — fast sanity check
df = pg_sql(db, f"""
    SELECT timestamp_utc, rp_story_id, relevance, event_sentiment_score,
           headline, event_text
    FROM ravenpack_dj.rpa_djpr_equities_2007
    WHERE rp_entity_id = '{rp_entity_id}'
      AND rpa_date_utc BETWEEN '2007-01-01' AND '2007-03-31'
    ORDER BY timestamp_utc
    LIMIT 50
""")

print(f"{len(df)} rows")
df.head()

In [ ]:
df["relevance_score"]       = pd.to_numeric(df["relevance"], errors="coerce") / 100
df["event_sentiment_score"] = pd.to_numeric(df["event_sentiment_score"], errors="coerce")
df["sentiment_score"]       = df["relevance_score"] * df["event_sentiment_score"]

print(f"Rows with sentiment_score : {df['sentiment_score'].notna().sum()} / {len(df)}")
print(f"Score range               : [{df['sentiment_score'].min():.3f}, {df['sentiment_score'].max():.3f}]")

# Show headline + event_text side by side for rows that have a sentiment score
sample = (
    df[["timestamp_utc", "headline", "event_text", "relevance_score", "event_sentiment_score", "sentiment_score"]]
    .dropna(subset=["sentiment_score"])
    .head(10)
)

# Print event_text in full so it's readable
pd.set_option("display.max_colwidth", None)
sample

In [ ]:
db.close()
print("Done.")